In [37]:
import os
import h5py
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [20]:
BASE_DIR = Path("../Datensatz/data-transfer-student")

In [29]:
def build_wsi_cohort(cohort_dir: Path, n_bins: int = 4) -> pd.DataFrame:
    df_clin = pd.read_csv(
        cohort_dir / "clinical/clinical_survival_per_slide.csv")
    wsi_dir = cohort_dir / "wsi_features/conch_v15"
    available_wsi = {p.stem for p in wsi_dir.glob("*.h5")}

    df_clin = df_clin.dropna(subset=['survival_months', 'censorship']).copy()
    df_clin = df_clin[df_clin['survival_months'] > 0].copy()
    df_clin['slide_id'] = df_clin['slide_id'].str.replace(
        '.svs', '', regex=False)
    df_clin = df_clin[df_clin["slide_id"].isin(available_wsi)].copy()

    
    df_clin["event"] = 1 - df_clin["censorship"] # 1 = Todesfall, 0 = zensiert


    df_clin["dx_num"] = df_clin["slide_id"].apply(
        # extrahiert die DX-Nummer aus dem slide_id String. Wenn keine gefunden wird, wird 999 gesetzt
        lambda s: int(re.search(r"-DX(\d+)\.", s).group(1)
                      ) if re.search(r"-DX(\d+)\.", s) else 999
    )
    df_clin["n_patches"] = df_clin["slide_id"].apply(
        # fügt eine neue Spalte n_patches hinzu, die die Anzahl der Patches in der HDF5-Datei für jeden Slide zählt
        lambda s: h5py.File(wsi_dir / f"{s}.h5", "r")["features"].shape[0]
    )
    df_clin = (
        df_clin.sort_values(by=["case_id", "dx_num", "n_patches"], ascending=[
                            True, True, False])
        .drop_duplicates(subset=["case_id"], keep="first")
        # da wir nur einen Slide pro Patient wollen, wird hier nach case_id gruppiert und der Slide mit der niedrigsten DX-Nummer und den meisten Patches behalten
        .reset_index(drop=True)
    )

    # 4. Diskrete Survival-Bins (4 Quartile auf Basis der Todesfälle / uncensored)
    # Extrahiert die Überlebenszeiten der Todesfälle
    uncensored_times = df_clin.loc[df_clin["event"] == 1, "survival_months"]
    # Teilt die Überlebenszeiten der Todesfälle in n_bins Quartile ein und gibt die Bin-Grenzen zurück
    _, q_bins = pd.qcut(uncensored_times, q=n_bins, retbins=True, labels=False)
    q_bins[0] = -1e-5  # Setzt die untere Grenze des ersten Bins auf einen kleinen negativen Wert, um sicherzustellen, dass alle Überlebenszeiten in den Bins enthalten sind
    # Setzt die obere Grenze des letzten Bins auf den maximalen Überlebenszeitwert + einen kleinen positiven Wert
    q_bins[-1] = df_clin["survival_months"].max() + 1e-5
    df_clin["time_bin"] = pd.cut(
        df_clin["survival_months"], bins=q_bins, labels=False, include_lowest=True)
    # Kombinierte Klasse aus Zensierung und Zeit-Bin (0..7) für exakte Stratifizierung
    df_clin["strat_label"] = df_clin["censorship"].astype(
        int) * n_bins + df_clin["time_bin"].astype(int)
    return df_clin

In [59]:
from sklearn.model_selection import StratifiedKFold

def cross_validate(df: pd.DataFrame, n_splits: int = 5) -> pd.DataFrame:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    df["fold"] = -1

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X=df, y=df["strat_label"])
    ):
        df.loc[val_idx, "fold"] = fold

    return df

In [ ]:
"""def cross_validate(df: pd.DataFrame, n_splits: int = 5) -> pd.DataFrame:

    # 5-Fold StratifiedGroupKFold direkt für jede Kohorte berechnen
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    df["fold"] = -1 # fügt eine neue Spalte "fold" hinzu, die standardmäßig auf -1 gesetzt ist. Diese Spalte wird später verwendet, um die Fold-Zuordnung für jeden Patienten zu speichern.

    for fold, (train_idx, val_idx) in enumerate(
        sgkf.split(X=df, y=df["strat_label"], groups=df["site"])
    ):
        df.loc[val_idx, "fold"] = fold # weist die Fold-Nummer für die Validierungsindizes in der "fold"-Spalte des DataFrames zu. Auf diese Weise wird jeder Patient einer bestimmten Fold-Nummer zugeordnet, basierend auf der Stratifizierung und Gruppierung, die durch StratifiedGroupKFold durchgeführt wurde.

    return df """

In [31]:
def build_dataset(cohort_dir: Path, n_bins: int = 4, n_splits: int = 5) -> pd.DataFrame:
    df_clin = build_wsi_cohort(cohort_dir, n_bins=n_bins)
    df_clin = cross_validate(df_clin, n_splits=n_splits)
    return df_clin

In [41]:
class SurvivalWSIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, wsi_dir: Path, max_patches: int = None):
        self.df = df.reset_index(drop=True)
        self.wsi_dir = Path(wsi_dir)
        self.max_patches = max_patches

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        h5_path = self.wsi_dir / f"{row['slide_id']}.h5"
        
        with h5py.File(h5_path, 'r') as f:
            features = f['features'][:]  # [M, 768] (CONCH v1.5)

        # Optionales Subsampling nur im Training (spart VRAM und beschleunigt Epochen)
        if self.max_patches is not None and len(features) > self.max_patches:
            indices = np.random.choice(len(features), self.max_patches, replace=False)
            features = features[indices]

        features_tensor = torch.tensor(features, dtype=torch.float32)
        
        target = torch.tensor([row['time_bin'], row['censorship']], dtype=torch.long)
        
        survival_months = torch.tensor(row['survival_months'], dtype=torch.float32)
        censorship = torch.tensor(row['censorship'], dtype=torch.float32)

        return features_tensor, target, survival_months, censorship

In [53]:
from sksurv.metrics import concordance_index_censored
from src.models.abmil import ABMIL

class NLLSurvLoss(nn.Module):
    """
    Discrete-Time Survival Loss (Negative Log-Likelihood).
    """
    def __init__(self, alpha: float = 0.15):
        super().__init__()
        self.alpha = alpha

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # target: [B, 2] -> Spalte 0 = time_bin, Spalte 1 = censorship (1 = zensiert, 0 = Event)
        y = target[:, 0].view(-1, 1)
        c = target[:, 1].view(-1, 1).float()

        hazards = torch.sigmoid(logits)
        S = torch.cumprod(1 - hazards, dim=1) # kumulatives Produkt: Tensor, bei dem jeder Eintrag die Wahrscheinlichkeit angibt, bis zu diesem spezifischen Zeitpunkt (und allen vorherigen) überlebt zu haben. 
        S_padded = torch.cat([torch.ones_like(c), S], dim=1)

        s_prev = torch.gather(S_padded, dim=1, index=y).clamp(min=1e-7)
        h_this = torch.gather(hazards, dim=1, index=y).clamp(min=1e-7)
        s_this = torch.gather(S_padded, dim=1, index=y + 1).clamp(min=1e-7)

        uncensored_loss = -(1 - c) * (torch.log(s_prev) + torch.log(h_this))
        censored_loss = -c * torch.log(s_this)

        neg_l = censored_loss + uncensored_loss
        loss = (1 - self.alpha) * neg_l + self.alpha * uncensored_loss
        return loss.mean()


In [69]:
def train_one_epoch(model, loader, loss_fn, optimizer, device, grad_accum_steps=8):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, (features, target, _, _) in enumerate(loader):
        features = features.to(device)  # [1, M, 768]
        target = target.to(device)      # [1, 2]

        results_dict, _ = model(h=features, loss_fn=loss_fn, label=target)
        loss = results_dict['loss'] / grad_accum_steps
        loss.backward()

        total_loss += results_dict['loss'].item()

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
            optimizer.step()
            optimizer.zero_grad()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    all_risk_scores = []
    all_censorships = []
    all_event_times = []

    for features, target, surv_months, censorship in loader:
        features = features.to(device)
        target = target.to(device)

        results_dict, _ = model(h=features, loss_fn=loss_fn, label=target)
        total_loss += results_dict['loss'].item()

        # Risk Score aus den 4 Survival-Bins berechnen
        hazards = torch.sigmoid(results_dict['logits'])
        survival = torch.cumprod(1 - hazards, dim=1)
        risk = -torch.sum(survival, dim=1)

        all_risk_scores.extend(risk.cpu().numpy())
        all_censorships.extend(censorship.numpy())
        all_event_times.extend(surv_months.numpy())

    # sksurv erwartet event_indicator = True für Todesfall (1 - censorship)
    event_indicators = (1 - np.array(all_censorships)).astype(bool)
    c_index = concordance_index_censored(
        event_indicators,
        np.array(all_event_times),
        np.array(all_risk_scores)
    )[0]

    return total_loss / len(loader), c_index, all_risk_scores

In [64]:
import json
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np

class ExperimentLogger:
    def __init__(self, base_dir: str, config: dict):
        self.config = config
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_name = f"{config['cohort']}_{config['model_type']}_{timestamp}"
        
        self.base_dir = Path(base_dir)
        self.run_dir = self.base_dir / run_name
        self.ckpt_dir = self.run_dir / "checkpoints"
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        
        # 1. Config als JSON speichern
        with open(self.run_dir / "config.json", "w") as f:
            json.dump(self.config, f, indent=4)
            
        self.epoch_logs = []
        self.oof_records = []

    def log_splits(self, df: pd.DataFrame):
        # 2. Exakte Splits und Bins sichern
        cols = ["case_id", "slide_id", "site", "survival_months", "censorship", "event", "time_bin", "fold"]
        available_cols = [c for c in cols if c in df.columns]
        df[available_cols].to_csv(self.run_dir / "dataset_splits.csv", index=False)

    def log_epoch(self, fold: int, epoch: int, train_loss: float, val_loss: float, val_c_index: float):
        # 3. Verlauf pro Epoche merken
        self.epoch_logs.append({
            "fold": fold,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_c_index": val_c_index
        })

    def log_oof_predictions(self, val_df: pd.DataFrame, risk_scores: list, fold: int):
        # 4. Beste Validierungs-Vorhersagen dieses Folds für spätere Kaplan-Meier-Analysen speichern
        oof_df = val_df[["case_id", "slide_id", "survival_months", "censorship", "event"]].copy()
        oof_df["fold"] = fold
        oof_df["risk_score"] = risk_scores
        self.oof_records.append(oof_df)

    def finalize(self, fold_c_indices: list):
        # Epochen-Logs & OOF-Predictions als CSV schreiben
        pd.DataFrame(self.epoch_logs).to_csv(self.run_dir / "metrics_per_epoch.csv", index=False)
        if self.oof_records:
            pd.concat(self.oof_records, ignore_index=True).to_csv(self.run_dir / "oof_predictions.csv", index=False)
        
        mean_c = float(np.mean(fold_c_indices))
        std_c = float(np.std(fold_c_indices))
        
        summary = {
            "run_name": self.run_dir.name,
            "mean_c_index": round(mean_c, 4),
            "std_c_index": round(std_c, 4),
            "fold_c_indices": [round(float(x), 4) for x in fold_c_indices],
            **self.config
        }
        
        # 5. Run-Summary speichern
        with open(self.run_dir / "summary.json", "w") as f:
            json.dump(summary, f, indent=4)
            
        # 6. An globale Master-CSV anhängen
        master_csv = self.base_dir / "experiment_summary.csv"
        summary_flat = {k: str(v) if isinstance(v, list) else v for k, v in summary.items()}
        df_row = pd.DataFrame([summary_flat])
        df_row.to_csv(master_csv, mode="a", header=not master_csv.exists(), index=False)
        
        print(f"\nAlle Ergebnisse archiviert unter: {self.run_dir}")

In [60]:
dataset = build_dataset(BASE_DIR / "TCGA-KIRC", n_bins=4, n_splits=5)

In [66]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Nutze Device: {device}")

Nutze Device: mps


In [70]:
# 1. Zentrale Konfiguration definieren
config = {
    "cohort": "TCGA-KIRC",
    "model_type": "abmil",
    "in_dim": 768,
    "embed_dim": 512,
    "num_fc_layers": 1,
    "dropout": 0.25,
    "attn_dim": 384,
    "gate": True,
    "num_classes": 4,
    "epochs": 20,
    "lr": 2e-4,
    "weight_decay": 1e-4,
    "grad_accum_steps": 8,
    "max_patches_train": None,
    "loss_alpha": 0.15,
    "seed": 42
}

logger = ExperimentLogger(base_dir="./runs", config=config)
logger.log_splits(dataset)

WSI_DIR = BASE_DIR / config["cohort"] / "wsi_features/conch_v15"

torch.manual_seed(config["seed"])
np.random.seed(config["seed"])

fold_results = []

for fold in range(5):
    print(f"\n==================== FOLD {fold} ====================")
    
    train_df = dataset[dataset['fold'] != fold]
    val_df = dataset[dataset['fold'] == fold]
    
    train_ds = SurvivalWSIDataset(train_df, wsi_dir=WSI_DIR, max_patches=config["max_patches_train"])
    val_ds = SurvivalWSIDataset(val_df, wsi_dir=WSI_DIR, max_patches=None)
    
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
    
    model = ABMIL(
        in_dim=config["in_dim"],
        embed_dim=config["embed_dim"],
        num_fc_layers=config["num_fc_layers"],
        dropout=config["dropout"],
        attn_dim=config["attn_dim"],
        gate=config["gate"],
        num_classes=config["num_classes"]
    ).to(device)
    
    loss_fn = NLLSurvLoss(alpha=config["loss_alpha"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])
    
    best_c_index = 0.0
    best_risk_scores = None
    
    for epoch in range(config["epochs"]):
        train_loss = train_one_epoch(
            model, train_loader, loss_fn, optimizer, device, grad_accum_steps=config["grad_accum_steps"]
        )
        val_loss, val_c_index, val_risk_scores = evaluate(model, val_loader, loss_fn, device)
        scheduler.step()
        
        # 1. In CSV-Log schreiben UND 2. live im Notebook ausgeben
        logger.log_epoch(fold, epoch + 1, train_loss, val_loss, val_c_index)
        print(f"Epoch {epoch+1:02d}/{config['epochs']:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val C-Index: {val_c_index:.4f}")
        
        if val_c_index > best_c_index:
            best_c_index = val_c_index
            best_risk_scores = val_risk_scores
            torch.save(model.state_dict(), logger.ckpt_dir / f"best_model_fold_{fold}.pt")
            
    print(f"--> Bester Val C-Index Fold {fold}: {best_c_index:.4f}")
    logger.log_oof_predictions(val_df, best_risk_scores, fold)
    fold_results.append(best_c_index)

print("\n==================================================")
print(f"5-Fold CV Ergebnis ({config['cohort']}): {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
logger.finalize(fold_results)


==================== FOLD 0 ====================
Epoch 01/20 | Train Loss: 0.9126 | Val Loss: 0.8536 | Val C-Index: 0.7111
Epoch 02/20 | Train Loss: 0.7413 | Val Loss: 0.8537 | Val C-Index: 0.7274
Epoch 03/20 | Train Loss: 0.7002 | Val Loss: 0.8337 | Val C-Index: 0.7324
Epoch 04/20 | Train Loss: 0.6863 | Val Loss: 0.8172 | Val C-Index: 0.7274
Epoch 05/20 | Train Loss: 0.6412 | Val Loss: 0.8028 | Val C-Index: 0.7588
Epoch 06/20 | Train Loss: 0.5668 | Val Loss: 0.7999 | Val C-Index: 0.7575
Epoch 07/20 | Train Loss: 0.4951 | Val Loss: 0.8096 | Val C-Index: 0.7638
Epoch 08/20 | Train Loss: 0.4850 | Val Loss: 0.8375 | Val C-Index: 0.7701
Epoch 09/20 | Train Loss: 0.4257 | Val Loss: 0.8939 | Val C-Index: 0.7437
Epoch 10/20 | Train Loss: 0.3860 | Val Loss: 0.9270 | Val C-Index: 0.7073
Epoch 11/20 | Train Loss: 0.3332 | Val Loss: 0.8737 | Val C-Index: 0.7010
Epoch 12/20 | Train Loss: 0.3172 | Val Loss: 0.9079 | Val C-Index: 0.6633
Epoch 13/20 | Train Loss: 0.2882 | Val Loss: 0.8890 | Val C-In

KeyboardInterrupt: 

In [72]:
torch.cuda.empty_cache()